# vbOCP — Sweep iperparametri PODNN (stato/aggiunto)

Notebook orchestratore bash-style: le celle lanciano solo comandi verso `src/rom/sweep_pod_nn.py`. Richiede che gli snapshot di training/test e la mesh siano gia' pronti (vedi `run_pipeline.ipynb`, sezioni 1-2).

## 1. Setup

In [ ]:
import os

while not os.path.isdir('src') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')
assert os.path.isdir('src'), "src/ non trovata: verifica dove e' montata la repo vbOCP"

REPO_ROOT           = os.getcwd()
CONFIG_PATH         = os.path.join('configs', 'test1.yaml')
SNAPSHOTS_DIR        = os.path.join('data', 'snapshots')
SNAPSHOTS_PATH       = os.path.join(SNAPSHOTS_DIR, 'test1_300.npz')
TEST_SNAPSHOTS_PATH  = os.path.join(SNAPSHOTS_DIR, 'test1_test150.npz')

os.makedirs(SNAPSHOTS_DIR, exist_ok=True)

print('REPO_ROOT          :', REPO_ROOT)
print('SNAPSHOTS_PATH      :', SNAPSHOTS_PATH)
print('TEST_SNAPSHOTS_PATH :', TEST_SNAPSHOTS_PATH)

## 2. Sweep sul numero di modi (hidden_dim fisso)

Mesh/snapshot vengono caricati una sola volta dentro lo script (non ad ogni punto). Con epoche piene e molti valori, richiede parecchio tempo.

In [ ]:
N_MODES_VALUES = '1,10,20,30,40,50,60,70,80,90,100,110,120,130,140,150'
HIDDEN_DIM = 32
EPOCHS = 50000
SWEEP_N_MODES_OUTPUT = os.path.join(SNAPSHOTS_DIR, 'sweep_n_modes.csv')

!python -m src.rom.sweep_pod_nn \
    --config {CONFIG_PATH} \
    --snapshots {SNAPSHOTS_PATH} \
    --test-snapshots {TEST_SNAPSHOTS_PATH} \
    --sweep-param n_modes \
    --values {N_MODES_VALUES} \
    --hidden-dim {HIDDEN_DIM} \
    --epochs {EPOCHS} \
    --output {SWEEP_N_MODES_OUTPUT}

In [ ]:
from pathlib import Path
from IPython.display import Image
Image(str(Path(SWEEP_N_MODES_OUTPUT).with_suffix('.png')))

In [ ]:
import csv

with open(SWEEP_N_MODES_OUTPUT) as f:
    for row in csv.DictReader(f):
        print(row)

## 3. Sweep su hidden_dim (N fisso)

*(da lanciare dopo aver scelto N dal risultato dello sweep sopra)*

In [ ]:
HIDDEN_DIM_VALUES = '16,32,64,128'
N_MODES_FIXED = 50  # aggiorna con il valore migliore trovato nello sweep precedente
SWEEP_HIDDEN_DIM_OUTPUT = os.path.join(SNAPSHOTS_DIR, 'sweep_hidden_dim.csv')

!python -m src.rom.sweep_pod_nn \
    --config {CONFIG_PATH} \
    --snapshots {SNAPSHOTS_PATH} \
    --test-snapshots {TEST_SNAPSHOTS_PATH} \
    --sweep-param hidden_dim \
    --values {HIDDEN_DIM_VALUES} \
    --n-modes {N_MODES_FIXED} \
    --epochs {EPOCHS} \
    --output {SWEEP_HIDDEN_DIM_OUTPUT}

In [ ]:
Image(str(Path(SWEEP_HIDDEN_DIM_OUTPUT).with_suffix('.png')))